# 03 Leakage Investigation

UCI Bank Marketing Dataset — cleaned data from `01_data_understanding_and_cleaning.ipynb`.

This notebook investigates data leakage and prediction-time feature availability. It does not train models, build pipelines, encode/scale variables, or engineer features. `duration` is investigated in depth but **not removed from the DataFrame** — that implementation step belongs to `04_feature_engineering.ipynb`. This notebook's output is a documented decision, not a modified dataset.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/bank_marketing_cleaned.csv')
df.shape

(41176, 21)

## 1. Prediction Scenario

> **Immediately before a planned customer contact, the bank uses all customer and campaign information already available at that moment to estimate the probability that the customer will subscribe to a term deposit, so that marketing resources can be prioritized.**

This fixes the prediction point precisely, and it's worth being exact about what it does and doesn't allow. The prediction happens **before the specific phone call that is about to happen** — but it can happen **after** earlier contacts in the same campaign have already taken place. In other words, this is *before a specific planned contact*, not *before the campaign starts*: if this is the third time the bank is calling a client, the prediction is made right before that third call, with the first two contacts already in the past. Every feature in this notebook is evaluated against one question: *would this value already exist, unchanged, at the moment the bank decides to place this specific call — distinguishing contacts that already happened from the one that hasn't yet?*

## 2. What Is Data Leakage?

Data leakage happens when a model has access, during training, to information that would not actually be available at the moment it needs to make a real prediction. The model still "works" on historical data — often suspiciously well — because the leaked feature encodes information correlated with, or a byproduct of, the very outcome being predicted.

**The important distinction this notebook is built around:**

| | A feature is predictive | A feature is available at prediction time |
|---|---|---|
| Question it answers | Does it correlate with / explain the target? | Does it exist, in a usable form, *before* the prediction has to be made? |
| How to check | Correlation, group differences, model importance | Trace the feature back to *when* in the process its value is actually determined |

These are independent properties. A feature can be highly predictive **and** available (ideal), weakly predictive **and** available (still fine, just not useful), or highly predictive **and unavailable** — which is the dangerous case, because it's the one that silently inflates offline metrics without producing a model that works in production. `duration`, investigated below, is a textbook example of the last case.

## 3. Feature Availability Timeline

Grouping features by *when* their value is actually determined, relative to the current call, gives a cleaner test than looking at each column in isolation:

1. **Before this campaign started** — outcomes and history from *previous* campaigns. Fixed and known long before this campaign's first call. → `previous`, `poutcome`, `pdays`
2. **Decided by the bank while planning this contact, or already completed earlier in this same campaign** — logistics the bank chooses, and the record of contacts already made to this client this campaign. Known the moment this specific call is scheduled, before dialing it. → `contact`, `month`, `day_of_week`, `campaign` (reasoned through in Section 5 — its dataset definition needs particular care, since it counts the current contact as well as prior ones)
3. **Generated by the current interaction itself** — cannot exist until the call happens and unfolds. → `duration`

Buckets 1 and 2 are available before the call by construction. Bucket 3 is the one that needs a decision — and it's a bucket of exactly one feature.

## 4. Investigation of `duration`

**What does it measure?** The duration, in seconds, of the last contact — i.e. *this* call, the one the prediction is meant to happen before.

**When does it become available?** Only once the call has connected and ended — duration is a measurement taken *of* the interaction, not a plan made *for* it. There is no way to know how long a phone call will last before making it.

**Would it be available before the current call?** No. By the time `duration` has a value, the call this project is trying to prioritize has already happened.

In [2]:
df.groupby('y')['duration'].describe().T

y,no,yes
count,36537.000000,4639.000000
mean,220.868079,553.256090
std,207.116178,401.190736
min,0.000000,37.000000
25%,95.000000,253.500000
50%,164.000000,449.000000
75%,279.000000,741.500000
max,4918.000000,4199.000000


Subscribers' calls run far longer on average (mean 553s / median 449s) than non-subscribers' (mean 221s / median 164s). That gap alone doesn't prove leakage — it's evidence the feature is predictive. The leakage argument comes from *when* that value is known, not from its size.

In [3]:
df.loc[df['duration'] == 0, ['duration', 'y']]

,duration,y
6250,0,no
23024,0,no
28055,0,no
33005,0,no


Every one of the 4 zero-duration calls has `y == 'no'`, with no exceptions. That's not a coincidence to explain statistically — it's a logical certainty: a call that never connects cannot end in a subscription. This is a useful edge case for making the general point precise: `duration` is **strongly associated with the target in the historical data** because the length of a conversation reflects how the interaction unfolded — a longer, more engaged conversation tends to coincide with a customer who is closer to saying yes. That association is real, and it says nothing yet about whether the feature can be used. The separate, decisive question is availability: for the BEFORE-CALL prediction scenario, the duration of the upcoming call is not known before that call takes place. Using it would mean giving the model information that is unavailable at the moment the prediction actually needs to be made.

In [4]:
target_numeric = df['y'].map({'no': 0, 'yes': 1})
print('correlation with target:', round(df['duration'].corr(target_numeric), 3))
print()
for threshold in [0, 60, 100, 180, 300, 500, 700, 1000]:
    subset = df[df['duration'] >= threshold]
    rate = (subset['y'] == 'yes').mean() * 100
    coverage = len(subset) / len(df) * 100
    print(f'duration >= {threshold:>4}: {coverage:5.1f}% of data, subscription rate {rate:5.2f}%')

correlation with target: 0.405

duration >=    0: 100.0% of data, subscription rate 11.27%
duration >=   60:  89.9% of data, subscription rate 12.54%
duration >=  100:  76.0% of data, subscription rate 14.57%
duration >=  180:  50.0% of data, subscription rate 19.81%
duration >=  300:  27.3% of data, subscription rate 27.84%
duration >=  500:  12.0% of data, subscription rate 42.50%
duration >=  700:   6.0% of data, subscription rate 53.22%
duration >= 1000:   2.3% of data, subscription rate 59.31%


This is the clearest demonstration of why the predictive-vs-available distinction from Section 2 matters here specifically. With no modeling at all — just a threshold on one column — the observed subscription rate climbs from the 11.27% baseline to 59.31% among calls lasting 1,000+ seconds, confirming `duration` is highly predictive. But that predictiveness cannot be exploited for the BEFORE-CALL scenario: the threshold can only be evaluated once a duration value exists, and a duration value only exists once the call this project is trying to prioritize has already happened. **Predictive is not the same property as available at prediction time**, and `duration` is a case where the two point in opposite directions.

**BEFORE-CALL model vs. AFTER-CALL model.** These are genuinely different prediction problems, not the same problem with an optional feature:

- A **BEFORE-CALL model** (this project's scope) predicts a probability using only information that exists prior to dialing, so it can be used to decide *who to call*. `duration` cannot be part of it.
- An **AFTER-CALL model** would predict or analyze outcomes using information from a call that has already happened — e.g. reviewing completed calls, or real-time agent guidance mid-call. `duration` would be legitimately available there. That is a different product with a different use case, and is out of scope for this project.

**Decision:** for the BEFORE-CALL scenario, `duration` is excluded from the feature set. It remains in `df` in this notebook — the exclusion is a documented policy decision to be implemented in `04_feature_engineering.ipynb`, not a deletion performed here.

## 5. Investigation of Campaign Features

`campaign` — number of contacts performed during the current campaign for this client, **including the current/last contact**, per the dataset documentation. This needs careful timing reasoning against the scenario defined in Section 1: the prediction happens immediately before *a specific planned contact*, which may not be the client's first contact this campaign. That means two different things can be true for the same row at once — contacts made **before** this one are settled history, while this specific contact has not happened yet — and `campaign`'s raw value does not distinguish between them.

In [5]:
df.groupby('y')['campaign'].describe().T

y,no,yes
count,36537.000000,4639.000000
mean,2.633385,2.051951
std,2.873768,1.666353
min,1.000000,1.000000
25%,1.000000,1.000000
50%,2.000000,2.000000
75%,3.000000,2.000000
max,56.000000,23.000000


Subscribers were contacted slightly *fewer* times on average this campaign (mean 2.05 vs 2.63) — consistent with the diminishing-returns pattern already found in notebook 02 (subscription rate falls from 13.0% at 1 contact to 4.6% at 7+).

**Reasoning about the exact prediction point.** Take a row where `campaign == 3`. Two readings are possible:

1. *"Two contacts have already been made this campaign, and a third is about to be attempted."* That count — 2 prior contacts — is genuinely known to the bank before dialing: it's the bank's own dialing log, not a measurement of how this call goes. No leakage.
2. *"Three contacts, including this one, have been made this campaign."* This is what the dataset actually stores, per its documentation. Read this way, the value `3` presumes the current call has already taken place — but at the exact moment defined in Section 1, that call hasn't happened yet.

These two readings differ by exactly one, and the raw `campaign` column as stored only gives reading 2. So `campaign` is not as clean as `previous`, `poutcome`, or `pdays` (Section 6): those describe *only* contacts before the current campaign, with no ambiguity. `campaign` mixes settled history (contacts 1 through N-1, safely available) with a count that only becomes N once the current, not-yet-made contact is included.

This is a materially different problem from `duration`, and it's worth being precise about why. `duration`'s value cannot be produced by *any* redefinition before the call happens — there is no way to know call length in advance, full stop. `campaign`'s raw value, by contrast, is off by a fixed, known amount: the number of contacts made *before* this one is simply `campaign - 1`, and that quantity is fully available before dialing. The unavailable part isn't the underlying information — it's specifically the raw column's inclusion of the not-yet-made contact.

**Decision:** `campaign` is **not excluded** — its underlying information (contacts made so far) is genuinely available before this specific planned contact, and it is not being changed merely for being predictive. But it also **cannot be used as stored** without a definitional adjustment: using the raw value as-is would count a contact that hasn't happened yet as if it had. This is flagged as a required adjustment for notebook 04 (feature engineering), not resolved here.

In [6]:
correlation = df['campaign'].corr(target_numeric)
print('campaign correlation with target:', round(correlation, 3))
df[['campaign', 'previous', 'pdays']].corr()['campaign']

campaign correlation with target: -0.066


campaign    1.000000
previous   -0.079182
pdays       0.052606
Name: campaign, dtype: float64

`campaign` correlates weakly with the target (-0.07) and only weakly with the prior-campaign history features — consistent with it describing something distinct: activity within the current campaign, not history from before it.

**`contact`, `month`, `day_of_week`** — all three describe *when and how* the bank plans to reach the client, decisions the bank makes while scheduling the call, not outcomes the customer produces. They are known at the moment a contact is scheduled, before dialing.

In [7]:
rate_by_contact = df.groupby('contact')['y'].apply(lambda s: (s == 'yes').mean() * 100).round(2)
rate_by_contact

contact
cellular     14.74
telephone     5.23
Name: y, dtype: float64

**Decision:** `contact` and `day_of_week` are available before the current call and are retained without reservation.

`month` is also a scheduling decision and is therefore available before the call in principle — but notebook 02 found it has the widest subscription-rate swing of any categorical feature (6.4%–50.6%), concentrated in months with much smaller sample sizes, and flagged that this dataset spans 2008–2010, so month may be entangled with *which* economic conditions were in effect rather than a pure calendar effect. That's not a leakage concern — the value is still known before the call — but it's a modeling caution worth carrying forward: `month`'s apparent signal should be checked against the economic indicators before being trusted as an independent effect.

**Decision:** `month` is available before the current call and is retained, flagged for caution rather than exclusion.

## 6. Investigation of Previous Campaign Features

`previous` (contacts before this campaign), `poutcome` (outcome of the previous campaign), and `pdays` (days since last contact from a previous campaign, `999` = sentinel) all describe **campaigns that concluded before the current one began**. By definition, that history is fixed and fully known before the current campaign's first call — there is no scenario in which this data wouldn't yet exist at prediction time.

In [8]:
pd.crosstab(df['previous'] == 0, df['poutcome'])

poutcome,failure,nonexistent,success
previous,,,
False,4252,0,1373
True,0,35551,0


`previous == 0` aligns exactly with `poutcome == 'nonexistent'` — internally consistent, confirming both describe the same underlying history.

In [9]:
mismatch = df[(df['previous'] > 0) & (df['pdays'] == 999)]
print('rows with previous > 0 but pdays == 999:', len(mismatch))
mismatch['poutcome'].value_counts()

rows with previous > 0 but pdays == 999: 4110


poutcome
failure    4110
Name: count, dtype: int64

This reproduces the inconsistency flagged in notebook 02: 4,110 clients were contacted before (`previous > 0`, `poutcome == 'failure'`) yet carry the `pdays == 999` sentinel — the day-count simply wasn't recorded for most past failures. It's important to be precise about what kind of problem this is: it is **not** a leakage issue. Every one of these 4,110 values still describes something that happened *before* the current campaign — the bank knew, before dialing, that this client had been contacted before and that it hadn't succeeded. What's unreliable here is the specific **day-count encoding**, not the **availability** of the underlying history. That's a data-quality/consistency problem for notebook 04's feature engineering to resolve carefully, not a reason to exclude `pdays`.

**Decision:** `previous`, `poutcome`, and `pdays` are all available before the current call and are retained. `pdays`'s sentinel inconsistency is carried forward as an engineering caution, separate from this notebook's leakage conclusion.

## 7. Feature Availability Assessment

Summarizing the investigation above for every feature examined in this notebook.

In [10]:
availability_table = pd.DataFrame([
    {
        'Feature': 'duration',
        'Meaning': 'Duration (seconds) of the current/last contact',
        'Available Before Current Call?': 'No',
        'Potential Leakage?': 'Yes',
        'Decision': 'Exclude',
        'Reason': 'Strongly associated with the target in historical data, but only exists once the current call has happened and ended; using it introduces information unavailable at prediction time.',
    },
    {
        'Feature': 'campaign',
        'Meaning': 'Number of contacts made this campaign for this client, including this one',
        'Available Before Current Call?': 'Partially - contacts before this one: yes. This specific contact: not yet, but the raw value counts it as done',
        'Potential Leakage?': 'Minor, fixable',
        'Decision': 'Retain (adjust in notebook 04)',
        'Reason': "Contacts made so far (campaign - 1) are genuinely known before dialing, but the raw stored value counts the current, not-yet-made contact too - needs a definitional adjustment before use, not exclusion.",
    },
    {
        'Feature': 'contact',
        'Meaning': 'Contact communication channel (cellular / telephone)',
        'Available Before Current Call?': 'Yes',
        'Potential Leakage?': 'No',
        'Decision': 'Retain',
        'Reason': 'Channel is chosen by the bank while planning the contact.',
    },
    {
        'Feature': 'month',
        'Meaning': 'Month the contact was made',
        'Available Before Current Call?': 'Yes',
        'Potential Leakage?': 'No',
        'Decision': 'Retain (caution)',
        'Reason': 'Scheduling decision, known in advance. Flagged in EDA for a very wide rate swing possibly entangled with the 2008-2010 economic timeline rather than a pure calendar effect.',
    },
    {
        'Feature': 'day_of_week',
        'Meaning': 'Day of week the contact was made',
        'Available Before Current Call?': 'Yes',
        'Potential Leakage?': 'No',
        'Decision': 'Retain',
        'Reason': 'Scheduling decision, known in advance; EDA found only a narrow, weak rate range.',
    },
    {
        'Feature': 'previous',
        'Meaning': 'Number of contacts performed before this campaign, for this client',
        'Available Before Current Call?': 'Yes',
        'Potential Leakage?': 'No',
        'Decision': 'Retain',
        'Reason': 'Describes campaigns that concluded before the current one; fully resolved before any current-campaign call.',
    },
    {
        'Feature': 'poutcome',
        'Meaning': 'Outcome of the previous marketing campaign',
        'Available Before Current Call?': 'Yes',
        'Potential Leakage?': 'No',
        'Decision': 'Retain',
        'Reason': 'Historical outcome from a prior, already-concluded campaign; perfectly consistent with previous == 0.',
    },
    {
        'Feature': 'pdays',
        'Meaning': 'Days since last contact in a previous campaign (999 = sentinel)',
        'Available Before Current Call?': 'Yes',
        'Potential Leakage?': 'No',
        'Decision': 'Retain (caution)',
        'Reason': 'Historical, pre-campaign information. The 999 sentinel is inconsistently recorded for ~97% of past failures (4,110 rows) - a data-quality issue for careful encoding in notebook 04, not a leakage issue.',
    },
])
availability_table

,Feature,Meaning,Available Before Current Call?,Potential Leakage?,Decision,Reason
0,duration,Duration (seconds) of the current/last contact,No,Yes,Exclude,Strongly associated with the target in histori...
1,campaign,Number of contacts made this campaign for this...,Partially - contacts before this one: yes. Thi...,"Minor, fixable",Retain (adjust in notebook 04),Contacts made so far (campaign - 1) are genuin...
2,contact,Contact communication channel (cellular / tele...,Yes,No,Retain,Channel is chosen by the bank while planning t...
3,month,Month the contact was made,Yes,No,Retain (caution),"Scheduling decision, known in advance. Flagged..."
4,day_of_week,Day of week the contact was made,Yes,No,Retain,"Scheduling decision, known in advance; EDA fou..."
5,previous,Number of contacts performed before this campa...,Yes,No,Retain,Describes campaigns that concluded before the ...
6,poutcome,Outcome of the previous marketing campaign,Yes,No,Retain,"Historical outcome from a prior, already-concl..."
7,pdays,Days since last contact in a previous campaign...,Yes,No,Retain (caution),"Historical, pre-campaign information. The 999 ..."


## 8. Final Feature Policy

**Scenario adopted:** BEFORE-CALL — estimate subscription probability before the customer is contacted, so marketing effort can be prioritized.

**Excluded — unavailable at prediction time:**
- `duration` — only exists after the current call has happened. This is the one feature this notebook concludes must be dropped for the defined scenario.

**Usable — available before the call, no reservations:**
- `contact`, `day_of_week`, `previous`, `poutcome` — plus every client-background feature examined in notebooks 01-02 (`age`, `job`, `marital`, `education`, `default`, `housing`, `loan`) and the five economic indicators (`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`) — none of these describe the current interaction, so none carry the leakage risk `duration` does.

**Usable with a required adjustment — genuinely available, but not as currently stored:**
- `campaign` — the number of contacts made *before* this one (`campaign - 1`) is available before this specific planned contact, but the raw stored value counts the current, not-yet-made contact as well. Not excluded — the underlying information is available and it is not being dropped merely for being predictive — but it must be adjusted to reflect prior contacts only before it is used, which is a notebook 04 task.

**Usable with caution — available, but with a modeling nuance to track:**
- `month` — available as a scheduling decision, but its strong rate swing may be entangled with the dataset's 2008-2010 economic timeline; worth checking against the economic indicators during feature engineering and interpretation, not excluding now.
- `pdays` — available and historical, but its `999` sentinel is inconsistently recorded for past failures; needs deliberate, careful encoding in notebook 04, not a naive `!= 999` check.

**On what this decision means:** dropping `duration` does not make the resulting model "better" in an absolute sense — a model trained with `duration` will likely show noticeably higher offline accuracy, precision, recall, and AUC than one without it. That difference is expected and doesn't indicate a modeling error. What excluding `duration` does is make the feature set **consistent with the prediction scenario this project defined**: a model meant to prioritize customers *before* they're called cannot depend on a value that only exists *after* the call. A high-performing BEFORE-CALL model and a high-performing AFTER-CALL model are solving different problems, and only the former matches this project's stated objective.

## 9. Key Findings

- **`duration` is excluded from the BEFORE-CALL feature set.** It is strongly associated with the target in the historical data — call length reflects how the interaction unfolded — but that association is a fact about predictiveness, not availability. All 4 zero-duration calls end in `y == 'no'`, and subscription rate climbs from an 11.27% baseline to 59.31% among calls of 1,000+ seconds using nothing but a threshold on one column, confirming just how predictive it is. None of that changes the timing fact that decides the outcome: the duration of the upcoming call is not known before that call takes place, so it is unavailable at the moment this project's prediction has to be made.
- **Predictive power and prediction-time availability are independent properties**, and `duration` is the clearest possible illustration: it is the single most correlated numeric feature with the target (+0.41) found across notebooks 01-03, and still cannot be used, because the scenario defines the prediction point as *before* the value exists.
- **`campaign` is retained, but flagged for a required adjustment rather than accepted as-is.** Distinguishing *before the campaign starts* from *before this specific planned contact* (Section 1) matters here specifically: contacts made before this one (`campaign - 1`) are genuinely known in advance, but the raw stored value counts the current, not-yet-made contact too. Unlike `duration`, this is a fixed, resolvable offset, not a fundamental unavailability — which is why the decision is "retain with adjustment," not "exclude."
- **`previous`, `poutcome`, and `pdays` are retained as historical, pre-campaign information.** The `pdays == 999` inconsistency for 4,110 past-failure clients (first found in notebook 02) was re-examined here and confirmed to be a data-quality/encoding problem, not a leakage problem — the underlying history is still fully known before the current call.
- **`month` is retained but flagged.** It's a scheduling decision, known in advance, but its unusually wide subscription-rate swing may partly reflect the dataset's 2008-2010 economic timeline rather than a pure seasonal effect.

### Carried forward to notebook 04
- Build the modeling feature set by excluding `duration` — this notebook documents the decision; the raw cleaned dataset and this notebook's `df` are unchanged.
- Adjust `campaign` to count only contacts made *before* the current one (conceptually `campaign - 1`) before using it as a feature — the raw column as stored includes the current, not-yet-made contact.
- Decide a deliberate, non-naive encoding for `pdays`'s `999` sentinel that accounts for the 4,110-row mismatch with `previous`/`poutcome`.
- When engineering or interpreting `month`, check its relationship to the economic indicators before treating its effect as independent.